In [1]:
import torch 
import numpy as np

In [2]:
path_to_data = "/work/scratch-pw5/bradlesc/climsim/processed/group_by_months/DJF/sample_rate_7/"
file_name = 'input.npy'

In [3]:
data = np.load(path_to_data + file_name)

In [4]:
data.shape

(2843904, 94)

In [5]:
# check for nan and inf
np.isnan(data).any(), np.isinf(data).any()

(np.False_, np.False_)

In [6]:
2843904/384

7406.0

In [33]:
x = np.arange(10)
x_repeated = np.repeat(x, 3).reshape(10, 3).transpose()

# add noise to each location
x_repeated = x_repeated + np.random.normal(0, 1, x_repeated.shape) * 1e-5

x_mean = x_repeated.mean(axis=0)
x_min = x_repeated.min(axis=0)
x_max = x_repeated.max(axis=0)
# print(x_mean)

# normalized_x = (x_repeated - x_mean) / (x_max - x_min)
# print(normalized_x)

print((x_repeated - x_mean)/(x_max - x_min))

[[-0.50124993 -0.12667971 -0.14452912 -0.66329724 -0.10314886 -0.51703757
   0.49069832  0.57018886  0.36146304 -0.40729467]
 [ 0.00249987  0.56333985  0.57226456  0.33670276  0.55157443  0.48296243
  -0.50930168 -0.42981114  0.27707393 -0.18541066]
 [ 0.49875007 -0.43666015 -0.42773544  0.32659447 -0.44842557  0.03407515
   0.01860335 -0.14037773 -0.63853696  0.59270533]]


In [34]:
import xarray as xr

ds = xr.open_dataset('/home/users/bradlesc/projects/ClimSim/preprocessing/normalizations/outputs/output_scale.nc')
ds

<xarray.Dataset> Size: 3kB
Dimensions:         (lev: 60)
Dimensions without coordinates: lev
Data variables: (12/14)
    ptend_t         (lev) float64 480B ...
    ptend_q0001     (lev) float64 480B ...
    ptend_q0002     (lev) float64 480B ...
    ptend_q0003     (lev) float64 480B ...
    ptend_u         (lev) float64 480B ...
    ptend_v         (lev) float64 480B ...
    ...              ...
    cam_out_PRECSC  float64 8B ...
    cam_out_PRECC   float64 8B ...
    cam_out_SOLS    float64 8B ...
    cam_out_SOLL    float64 8B ...
    cam_out_SOLSD   float64 8B ...
    cam_out_SOLLD   float64 8B ...

In [36]:
path_to_output_scaling = '/home/users/bradlesc/projects/ClimSim/preprocessing/normalizations/outputs/output_scale.nc'
v1_targets = ['ptend_t','ptend_q0001','cam_out_NETSW','cam_out_FLWDS','cam_out_PRECSC','cam_out_PRECC','cam_out_SOLS','cam_out_SOLL','cam_out_SOLSD', "cam_out_SOLLD"]
levels = 45

In [ ]:
def process_output_scaling(path_to_scaling_file: str, target_variables: list, levels: int) -> np.ndarray:
    """
    Loads the output scaling, selects target variables and levels of interest and returns a np arry of shape (features,)
    """
    max_levels = 60
    min_levels = max_levels - levels
    out_scale = xr.open_dataset(path_to_scaling_file)
    out_scale = out_scale[list(target_variables)]
    out_scale = out_scale.sel(lev=slice(min_levels, max_levels))
    out_scale_array = out_scale.to_array()

    v_vectors = out_scale_array[0:2]

    v_scalars = out_scale_array[2:, 0]

    out_scale = np.concatenate([v_vectors.values.flatten(), v_scalars.values.flatten()])
    
    return out_scale


    
output_scaling = process_output_scaling(path_to_output_scaling, v1_targets, levels)

print(output_scaling.shape)
print(output_scaling)

(98,)
[1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03
 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03
 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03
 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03
 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03
 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03
 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03 1.00464e+03
 1.00464e+03 1.00464e+03 1.00464e+03 2.83470e+06 2.83470e+06 2.83470e+06
 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06
 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06
 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06
 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06
 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e+06
 2.83470e+06 2.83470e+06 2.83470e+06 2.83470e

In [47]:
x = np.arange(10)
x_repeated = np.repeat(x, 3).reshape(10, 3).transpose()
print(x_repeated)
sf = np.arange(11, 21)
print(sf.shape)
print(sf)

x_scaled = x_repeated * sf
print(x_scaled.shape)
print(x_scaled)

[[0 1 2 3 4 5 6 7 8 9]
 [0 1 2 3 4 5 6 7 8 9]
 [0 1 2 3 4 5 6 7 8 9]]
(10,)
[11 12 13 14 15 16 17 18 19 20]
(3, 10)
[[  0  12  26  42  60  80 102 126 152 180]
 [  0  12  26  42  60  80 102 126 152 180]
 [  0  12  26  42  60  80 102 126 152 180]]
